# Arabic Audio Transcriber — Nature Video Edition

This is a separate version of `transcribe2.ipynb`. Transcription, model caching,
confidence alerts, translation files, redaction, and the SRT-only converter share the local pipeline helpers. The final-video section has been replaced.

**Workflow**
1. Verify dependencies, configure the model, and batch transcribe with **Cells 1–3** as before.
2. Translate externally with translation_prompt.md and fill the matching
   `transcript/english/<name>.txt` files, keeping their timestamp boundaries.
3. Run **Cell 8A** (video settings) and **Cell 8B** (helpers), then **Cell 8D**
   to make the final videos in `output/nature/<name>_final.mp4`.
4. **Cell 8C** is an optional silent design preview. It works with blank translation files.
   Run **8A → 8B → 8C** directly to inspect the style without running transcription.
   **Cell 9** still converts English transcripts to SRT without rendering.

Videos start immediately with random nature excerpts and the original media's
audio; the intro is **off by default**. Set `INCLUDE_INTRO = True` in Cell 8A to
prepend `INTRO_PATH` with its audio. Nature-clip audio is discarded. Captions keep
their original timings (shifted by the intro's encoded duration only when enabled).
The final image fades to black. Subtitle background color, opacity and padding are configurable in Cell 8A.

**Prerequisites:** the original Python packages (`pip install -r requirements.txt`)
and a full [FFmpeg build](https://ffmpeg.org/download.html) with libass and ffprobe
on PATH. Rendering adds no Python packages. Use Python 3.11 or newer. Open this
notebook with the `youtube-translator` project folder as Jupyter's working folder.


Use `translation_prompt.md` for manual translation or `codex/README.md` for folder-based translation. Keep meanings inside their original timestamp blocks.


In [ ]:
# Cell 1: Verify dependencies. Open Jupyter from the repository folder.
import importlib.util
import sys
missing = [name for name in ("faster_whisper", "torch") if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError("Install the local requirements in this notebook's environment: python -m pip install -r requirements.txt")
from pathlib import Path
if not Path("transcriber_utils.py").is_file():
    raise RuntimeError("Start Jupyter from the repository folder so the shared modules can be imported.")
import torch
from video_rendering import find_ffmpeg
print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "Using CPU transcription.")
try:
    ffmpeg, ffprobe = find_ffmpeg()
    print(f"Rendering tools ready: {ffmpeg} and {ffprobe}")
except RuntimeError as error:
    print(f"Rendering setup: {error} Transcription can still be run separately.")


## Configuration
Set the model, language, optional start/end times, and quality-report thresholds below.
The selected time range applies to every file; timestamps still refer to the original media.
CPU uses int8; CUDA uses float16. Available model names are described in README.md.


In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
import torch
from pathlib import Path

MODEL_SIZE = "large-v3-turbo"   # tiny | base | small | medium | large-v2
LANGUAGE   = "ar"          # e.g. "ar", "en", "fr", or None to auto-detect

# Optional time range — set to None to transcribe the whole file
START_TIME = None          # e.g. "8:00" or "1:25:00"
END_TIME   = None          # e.g. "25:00"

# ── Confidence alerting during transcription ─────────────────────────────────
# Flags audio the model could not really transcribe (loud static, distortion,
# heavy noise, music). Results go to transcript/arabic/<name>.flagged.txt.
FLAG_LOW_CONFIDENCE = True   # False disables all confidence checks below

# PRIMARY check — the one that catches static/distortion. When audio is
# unintelligible, Whisper does not report low confidence; it silently collapses
# the whole stretch into one long segment holding a few words. So the giveaway
# is text density: characters of transcript per second of audio. This is
# compared against the file's own median, so it adapts to language and pace.
STALL_MIN_SECONDS = 8.0     # only judge segments at least this long
STALL_CPS_RATIO   = 0.25    # flag when chars/sec falls below this × file median

# Long stretches with no transcript at all — silence, or audio the VAD threw away.
GAP_MIN_SECONDS = 20.0

# SECONDARY signals. These are decode-window-level: every segment in a ~30 s
# window shares one value, so they are coarse and are grouped into a single
# alert per window rather than repeated per line.
LOW_AVG_LOGPROB        = -0.55  # healthy audio typically sits around -0.1 … -0.3
HIGH_COMPRESSION_RATIO = 3.0    # 2.4-2.6 is normal for lists/repetitive speech
HIGH_NO_SPEECH_PROB    = 0.6    # only meaningful when vad_filter is off (VAD zeroes it)

# Arabic transcripts from Cell 3 land here; blank English stubs are created alongside
TRANSCRIPTS_DIR = Path("transcript/arabic")   # Arabic [MM:SS-MM:SS] transcripts
ENGLISH_DIR     = Path("transcript/english")  # blank stubs → fill with translations

# ── Helpers ───────────────────────────────────────────────────────────────────
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

from transcriber_utils import parse_timestamp as _parse_ts, format_timestamp as _fmt_ts, validate_time_range
start_sec, end_sec = validate_time_range(START_TIME, END_TIME)

print(f"Model   : {MODEL_SIZE}  |  Device: {DEVICE}  |  Compute: {COMPUTE_TYPE}")
print(f"Language: {LANGUAGE or 'auto-detect'}")
if start_sec is not None or end_sec is not None:
    s = _fmt_ts(start_sec or 0)
    e = _fmt_ts(end_sec) if end_sec is not None else "end"
    print(f"Range   : {s} → {e}")
else:
    print("Range   : full file")


## 📦 Model cache

Downloads `MODEL_SIZE` from Hugging Face **once** and reuses the local cache on every later run — no re-download, no network call, unless you explicitly ask to check for updates.

In [ ]:
# ── Cell 2B: Model cache — download once, reuse offline ──────────────────────
#
# First run of a new MODEL_SIZE: downloads the model (this is the slow part —
# install `hf_xet` per Cell 1 to speed it up).
# Every later run: loads straight from the local Hugging Face cache with
# local_files_only=True, so there is zero network traffic and zero delay.
#
# Set CHECK_FOR_MODEL_UPDATES = True and re-run this cell to check Hugging Face
# for a newer revision of the model (only re-downloads if something changed).

from pathlib import Path
from faster_whisper.utils import download_model

CHECK_FOR_MODEL_UPDATES = False


def _is_complete(model_dir) -> bool:
    """local_files_only=True happily returns a snapshot dir even if the big
    weights file never finished downloading (e.g. a previous run was
    interrupted). Verify model.bin actually resolves before trusting it."""
    p = Path(model_dir) / "model.bin"
    return p.exists() and p.stat().st_size > 0


MODEL_PATH = None
if not CHECK_FOR_MODEL_UPDATES:
    try:
        candidate = download_model(MODEL_SIZE, local_files_only=True)
        if _is_complete(candidate):
            MODEL_PATH = candidate
            print(f"✓ Model '{MODEL_SIZE}' ready (local cache): {MODEL_PATH}")
        else:
            print(f"'{MODEL_SIZE}' is cached but incomplete (missing/empty model.bin) — will re-download.")
    except Exception:
        pass

if MODEL_PATH is None:
    print(f"Downloading '{MODEL_SIZE}' (one-time — this is the slow part; hf_xet speeds it up)…")
    MODEL_PATH = download_model(MODEL_SIZE, local_files_only=False)
    if not _is_complete(MODEL_PATH):
        raise RuntimeError(f"Download finished but model.bin is still missing/empty in {MODEL_PATH}")
    print(f"✓ Downloaded and cached: {MODEL_PATH}")

## 📥 Batch transcription

Set `INPUT_FOLDER` in the cell below to a folder of audio/video files. Every file in the folder is transcribed **in succession**. Arabic transcripts and English stubs are written as:

```
transcript/arabic/<name>.txt    ← Arabic [MM:SS-MM:SS] transcript
transcript/english/<name>.txt   ← blank stub (fill with the translation)
```

Filenames match the media exactly (`test.mp3` → `transcript/arabic/test.txt` + `transcript/english/test.txt`). Translate externally (prompt at the top of this notebook), put the English into the matching stub under `transcript/english/`, then run the final-video cell at the bottom.

**Audio-quality alerting.** After each file, the cell checks whether any stretch of audio failed to transcribe properly and writes `transcript/arabic/<name>.flagged.txt`:

- **SEVERE** — a long stretch that produced almost no text. This is what static, distortion, heavy noise, or music looks like: Whisper does *not* report low confidence, it quietly collapses the whole stretch into one segment holding a few words. The transcript there is missing or wrong, so re-record or clean the audio before translating.
- **warning** — a long stretch with no transcript at all (usually just silence), or a weak decode signal worth a glance.

A red banner appears at the end of the batch if anything severe was found. Thresholds live in the configuration cell; note that a file which is *skipped* because its transcript already exists is not re-analysed — set `OVERWRITE = True` to regenerate a report.

In [ ]:
# ── Cell 3: Batch transcribe a folder ─────────────────────────────────────────
#
# Transcribes every audio/video file in INPUT_FOLDER, one after another, and
# saves Arabic timestamped transcripts in TRANSCRIPTS_DIR plus a blank English
# stub (same filename) in ENGLISH_DIR:
#   transcript/arabic/<name>.txt   (test.mp3 → Arabic transcript)
#   transcript/english/<name>.txt  (blank stub for the translation)
#
# Files that already have an Arabic transcript are skipped — set OVERWRITE = True
# to re-transcribe everything. Blank English stubs are only created if missing
# (existing translations are never overwritten).

from transcriber_utils import (confidence_alerts, format_cue, list_media, write_confidence_report)

from faster_whisper import WhisperModel

INPUT_FOLDER = r"input"   # ← folder containing the audio/video files
OVERWRITE    = False      # True = re-transcribe files that already have a transcript

def _ensure_english_stub(stem: str) -> Path:
    """Create transcript/english/<stem>.txt if it does not already exist."""
    ENGLISH_DIR.mkdir(parents=True, exist_ok=True)
    eng_txt = ENGLISH_DIR / f"{stem}.txt"
    if not eng_txt.exists():
        eng_txt.write_text("", encoding="utf-8")
        print(f"  ✓ Created blank English stub → {eng_txt}")
    return eng_txt


in_dir = Path(INPUT_FOLDER)
if not in_dir.is_dir():
    raise FileNotFoundError(f"Folder not found: {in_dir.resolve()}")

media_files = list_media(in_dir)  # Reject ambiguous stems before loading the model.
if not media_files:
    raise FileNotFoundError(f"No audio/video files found in: {in_dir.resolve()}")

TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
ENGLISH_DIR.mkdir(parents=True, exist_ok=True)

print(f"Folder      : {in_dir.resolve()}")
print(f"Arabic      : {TRANSCRIPTS_DIR.resolve()}")
print(f"English     : {ENGLISH_DIR.resolve()}")
print(f"Files       : {len(media_files)}")
for f in media_files:
    print(f"  • {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

# ── Load model once for the whole batch ───────────────────────────────────────
print(f"\nLoading Faster-Whisper '{MODEL_SIZE}' on {DEVICE} ({COMPUTE_TYPE}) …")
model = WhisperModel(MODEL_PATH, device=DEVICE, compute_type=COMPUTE_TYPE)

# Optional time range from Cell 2 (applies to every file in the batch)
use_clip   = start_sec is not None or end_sec is not None
clip_start = start_sec or 0.0
if not use_clip:
    clip_timestamps = "0"
    vad_filter = True
elif end_sec is None:
    clip_timestamps = str(clip_start)
    vad_filter = False
else:
    clip_timestamps = f"{clip_start},{float(end_sec)}"
    vad_filter = False

# ── Transcribe each file in succession ────────────────────────────────────────
done, skipped, failed = [], [], []
flagged_files = []   # [(filename, num_flagged_segments), …] — low-confidence alerting

for idx, media in enumerate(media_files, 1):
    out_txt = TRANSCRIPTS_DIR / f"{media.stem}.txt"
    print(f"\n{'─' * 70}\n[{idx}/{len(media_files)}] {media.name}")

    if out_txt.exists() and not OVERWRITE:
        print(f"  ↷ Skipped — transcript already exists: {out_txt}")
        _ensure_english_stub(media.stem)
        skipped.append(media.name)
        continue

    try:
        segments, info = model.transcribe(
            str(media),
            language=LANGUAGE,
            beam_size=5,
            temperature=0.0,
            condition_on_previous_text=False,
            vad_filter=vad_filter,
            clip_timestamps=clip_timestamps,

            # Helpful for long-form transcription
            word_timestamps=False,
            compression_ratio_threshold=2.4,
            log_prob_threshold=-1.0,
            no_speech_threshold=0.6,
        )

        duration = getattr(info, "duration", None)
        if duration:
            print(f"  Audio length: {_fmt_ts(duration)}")

        timed_lines = []
        seg_rows = []
        for seg in segments:
            text = seg.text.strip()
            if not text:
                continue
            start, end = float(seg.start or 0), float(seg.end or 0)
            line = format_cue(start, end, text)
            timed_lines.append(line)
            print(f"  {line}")

            dur = max(0.0, end - start)
            seg_rows.append({
                "start": start, "end": end, "dur": dur,
                "nchars": len(text),
                # Text density is the signal that exposes unintelligible audio.
                "cps": (len(text) / dur) if dur > 0 else 0.0,
                "alp": seg.avg_logprob,
                "ns": seg.no_speech_prob,
                "cr": seg.compression_ratio,
                "text": text,
            })

        out_txt.write_text("\n".join(timed_lines), encoding="utf-8")
        (TRANSCRIPTS_DIR / f"{media.stem}.plain.txt").write_text(
            " ".join(row["text"] for row in seg_rows), encoding="utf-8")
        print(f"  ✓ Saved {len(timed_lines)} segments → {out_txt}")
        _ensure_english_stub(media.stem)

        # ── Confidence report ────────────────────────────────────────────────
        alerts = []
        if FLAG_LOW_CONFIDENCE:
            cov_end = min(float(end_sec), duration) if end_sec is not None and duration else (end_sec or duration or 0.0)
            alerts = confidence_alerts(seg_rows, clip_start, cov_end,
                STALL_MIN_SECONDS=STALL_MIN_SECONDS, STALL_CPS_RATIO=STALL_CPS_RATIO,
                GAP_MIN_SECONDS=GAP_MIN_SECONDS, LOW_AVG_LOGPROB=LOW_AVG_LOGPROB,
                HIGH_COMPRESSION_RATIO=HIGH_COMPRESSION_RATIO, HIGH_NO_SPEECH_PROB=HIGH_NO_SPEECH_PROB)

        flagged_txt = TRANSCRIPTS_DIR / f"{media.stem}.flagged.txt"
        write_confidence_report(flagged_txt, media.name, alerts)
        if alerts:
            severe = [a for a in alerts if a["severity"] == "severe"]
            for a in alerts:
                mark = "⚠⚠" if a["severity"] == "severe" else " ⚠"
                print(f"  {mark} [{_fmt_ts(a['start'])}-{_fmt_ts(a['end'])}] {a['why']}")
            print(f"  → {len(severe)} severe, {len(alerts) - len(severe)} warning(s) — details in {flagged_txt}")
            flagged_files.append((media.name, len(severe), len(alerts) - len(severe)))
        else:
            print("  No alerts detected." if FLAG_LOW_CONFIDENCE else "  Quality checks disabled.")

        done.append(media.name)

    except Exception as e:
        print(f"  ✗ FAILED: {e}")
        failed.append(media.name)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'═' * 70}")
print(f"✓ Transcribed: {len(done)}   ↷ Skipped: {len(skipped)}   ✗ Failed: {len(failed)}")
for name in failed:
    print(f"  ✗ {name}")
print(f"Arabic transcripts : {TRANSCRIPTS_DIR.resolve()}")
print(f"English stubs      : {ENGLISH_DIR.resolve()}")

# ── Low-confidence alert ────────────────────────────────────────────────────
if flagged_files:
    n_severe = sum(s for _, s, _ in flagged_files)
    n_warn   = sum(w for _, _, w in flagged_files)
    bad_files = [f for f in flagged_files if f[1]]
    print(f"\n⚠ Audio quality: {n_severe} severe, {n_warn} warning(s) across {len(flagged_files)} file(s)")
    for name, s, w in flagged_files:
        mark = "⚠⚠" if s else " ⚠"
        print(f"   {mark} {name}: {s} severe, {w} warning(s)"
              f" — see {TRANSCRIPTS_DIR / (Path(name).stem + '.flagged.txt')}")
    try:
        from IPython.display import display, HTML
        if n_severe:
            display(HTML(
                "<div style='background:#ffdddd;border:2px solid #cc0000;padding:10px;"
                "border-radius:6px;font-weight:bold;color:#990000;font-size:14px;'>"
                f"⚠ {n_severe} region(s) of unusable audio in "
                f"{len(bad_files)} file(s): {', '.join(n for n, s, _ in bad_files)}<br>"
                "Likely static, distortion, or heavy noise — the transcript for those "
                "stretches is missing or wrong. Check the *.flagged.txt files before translating."
                "</div>"
            ))
        else:
            display(HTML(
                "<div style='background:#fff6dd;border:2px solid #cc9900;padding:10px;"
                "border-radius:6px;color:#805500;'>"
                f"⚠ {n_warn} minor warning(s) — no unusable audio detected. "
                "See the *.flagged.txt files if you want details.</div>"
            ))
    except Exception:
        pass
else:
    print("\nNo alerts in analyzed files. Skipped files were not rechecked." if FLAG_LOW_CONFIDENCE else "\nQuality checks were disabled.")

if failed:
    raise RuntimeError(f"Transcription failed for: {', '.join(failed)}")


## 🔇 Redact bad-audio regions

Takes the regions flagged above and — **only for the ones you approve** — deletes the subtitle lines inside them and puts a single line in their place saying the audio cuts out.

Run the cell below with `STEP` set to each of these in turn:

| `STEP` | What it does |
|---|---|
| `"review"` | Writes the review file `transcript/redactions.txt`; leaves transcripts unchanged. |
| `"review-interactive"` | Walks region by region, shows the exact lines that would be removed, and asks y/n. |
| `"apply-dry"` | Shows exactly what would change. Still writes nothing. |
| `"apply"` | Writes the changes. Originals are kept as `<name>.txt.bak`. |

Instead of `"review-interactive"` you can open `transcript/redactions.txt` and flip `approve: no` → `approve: yes` by hand. That file is also where you retime a region, reword its message, or add a stanza for a bad patch the detector missed — your edits survive re-running `"review"`.

**If speech resumes partway through a region:** a flagged region is normally *one* transcript line spanning the whole bad stretch, so redacting is all-or-nothing for that line — narrowing `region:` cannot save the recovered words; it will either still drop the whole line or leave the whole line (garbled lead-in included) untouched. To actually keep them, add a `keep:` line to that stanza:

```
region: 05:11-07:11
approve: yes
message: [Audio cuts out — this section could not be transcribed]
keep: 07:02-07:11 | إلى أن يموت طيب دية المرأة على النفس من دية الرجل
```

This carves `07:02-07:11` out of the placeholder and inserts it verbatim with its own timestamp — the message then only covers the genuinely bad `05:11-07:02`. A region can have more than one `keep:` line. Run `helper_scripts/probe_region.py <media> <start> <end>` first to find exactly where recoverable speech begins — it re-transcribes just that window with word-level timestamps, so you're not guessing at the split point.

Both `transcript/arabic/` and `transcript/english/` are updated when present. Redacting the Arabic **before** translating keeps the garbled text out of the translation and carries the placeholder through to the subtitles; redacting the English afterwards works too. Re-applying unchanged decisions is a no-op. If `keep:` contains Arabic recovery text, apply to Arabic before translating; do not insert it into an existing English translation.

The same tool works from a terminal:

```
python helper_scripts/redact_flagged.py review --interactive
python helper_scripts/redact_flagged.py apply --dry-run
python helper_scripts/redact_flagged.py apply
```


In [ ]:
# ── Cell 4: Redact approved bad-audio regions ────────────────────────────────
#
# Work through STEP in order: "review" → approve → "apply-dry" → "apply".
# Approving is either "review-interactive" here, or editing
# transcript/redactions.txt by hand (flip `approve: no` to `approve: yes`).

import importlib
import sys

if "helper_scripts" not in sys.path:
    sys.path.insert(0, "helper_scripts")
import redact_flagged as rf
importlib.reload(rf)   # so edits to the script take effect without a restart
rf.ARABIC_DIR = TRANSCRIPTS_DIR.resolve()
rf.ENGLISH_DIR = ENGLISH_DIR.resolve()

STEP   = "review"   # review | review-interactive | apply-dry | apply
ONLY   = None       # e.g. "20" to work on a single transcript, None = all
TARGET = "both"     # both | arabic | english

# Wording of the replacement subtitle. Only applies to regions added from here
# on; regions already in transcript/redactions.txt keep the message stored there.
rf.DEFAULT_MESSAGE = "[Audio cuts out — this section could not be transcribed]"

_steps = {
    "review":             lambda: rf.review(interactive=False, only=ONLY, target=TARGET),
    "review-interactive": lambda: rf.review(interactive=True,  only=ONLY, target=TARGET),
    "apply-dry":          lambda: rf.apply(dry_run=True,  only=ONLY, target=TARGET),
    "apply":              lambda: rf.apply(dry_run=False, only=ONLY, target=TARGET),
}
if STEP not in _steps:
    raise ValueError(f"STEP must be one of {list(_steps)}, got {STEP!r}")
_steps[STEP]()

## 🎞️ Nature-video rendering

The files still match by name: `input/test.mp3` ↔ `transcript/english/test.txt`.
Blank translation stubs and already finished videos are skipped. Set `ONLY` to a
filename stem for one video; `OVERWRITE_FINAL=True` recreates existing results.
This version saves into **`output/nature/`**, leaving your earlier exports separate.

The defaults follow the reference images: 1920×1080, centered white Times New
Roman captions on opaque black boxes with automatic line wrapping, gently blurred and darkened,
slightly muted scenery, and **ibn_adam** centered at 87% of the frame height.
The filter is applied before the text, so the text remains sharp. Settings in
8A control the font, size, subtitle background, blur, darkness, color and watermark position.
`SUBTITLE_BACKGROUND_COLOR` accepts a standard `#RRGGBB` color;
`SUBTITLE_BACKGROUND_OPACITY` ranges from `0.0` (transparent) to `1.0` (opaque).
`SUBTITLE_BACKGROUND_MODE="block"` draws a single rectangle around the whole
caption, including automatically wrapped lines, so translucent backgrounds have
uniform opacity with no darker overlap seams. This is the recommended mode.
Use `"line"` for the original notebook's per-line boxes; with padding and partial
opacity, those boxes can overlap. `SUBTITLE_BACKGROUND_PADDING` controls box
padding in either mode; `SUBTITLE_BACKGROUND_ENABLED=False` removes the boxes.
The renderer uses libass's [single-caption background style](https://github.com/libass/libass/wiki/Libass%27-ASS-Extensions#borderstyle4).
The intro is disabled by default; use `INCLUDE_INTRO=True` to add `INTRO_PATH`.
Use an installed font with the characters you need; libass uses fallback fonts
for Arabic and other scripts. Mixed Arabic/English text is shaped by libass.

**Randomization:** each nature excerpt is 10–30 seconds by default, including the
last one. The source file, excerpt duration and source in/out points are random.
The planner balances the durations across the whole audio track, avoiding an
undersized final shot. It avoids adjacent repeats whenever another eligible
source exists and favors less-used clips. Sources too short for the minimum are
reported and skipped; sources are never looped to manufacture a long shot.
Long programs can reuse source clips with new random in/out points.

Lengths and cuts are measured in output frames. Normally the nature section ends
with the audio to within one frame. For audio shorter than 10 seconds (or a
custom minimum/maximum range that cannot exactly fit), the last shot finishes
and silence pads the short tail, so every shot still meets the bounds.
The intro is separate from the 10–30 second nature-excerpt rule.

Set `RANDOM_SEED` to an integer to repeat a batch, or leave it `None` for a fresh
mix. Every finished video has a sibling **`<name>_final.clips.json`** recording its
seed, source files, source in/out points, output timeline and appearance settings.
Keep the files, settings and output frame rate unchanged to reproduce a mix.

Rendering checks that an encoder actually works, preferring NVENC and falling
back to CPU H.264. One excerpt is encoded at a time; the final assembly copies
the encoded video and encodes the audio once. Temporary files are cleaned up,
and a finished filename is only replaced after the encode and duration checks
succeed. Free disk space is needed for the temporary excerpts plus the output.

Implementation references: [FFmpeg video filters](https://www.ffmpeg.org/ffmpeg-filters.html)
and [FFmpeg concat demuxer](https://ffmpeg.org/ffmpeg-formats.html#concat-1).

**Skipping completed exports:** with `OVERWRITE_FINAL=False`, the batch skips a
pair if either `<name>_final.mp4` or `<name>_final.clips.json` is present in
`OUTPUT_FOLDER`. Keep the completion JSON when you rename a finished video so
the batch remembers it. Failed exports without either file are retried.
`OVERWRITE_FINAL=True` bypasses both checks to force a fresh render.


In [ ]:
# Cell 8A: Nature-video settings. This section also runs without transcription.
from pathlib import Path

MEDIA_FOLDER = r"input"
TRANSCRIPT_FOLDER = r"transcript/english"
NATURE_FOLDER = r"assets/nature"
OUTPUT_FOLDER = r"output/nature"       # separate from the original notebook's output
INCLUDE_INTRO = False                 # True adds the intro; False starts with nature footage
INTRO_PATH = r"assets/Intro.mp4"       # used only when INCLUDE_INTRO=True; None also omits it
OVERWRITE_FINAL = False
ONLY = None                          # a media filename stem, or None for the whole batch

MIN_CLIP_SECONDS = 8.0
MAX_CLIP_SECONDS = 20.0
RANDOM_SEED = None                   # fresh mix each run; an integer repeats the same mix
FADE_SECONDS = 1.5
TARGET_W, TARGET_H, TARGET_FPS = 1920, 1080, 24

# Background only: subtitles and watermark stay sharp. Set BLUR_SIGMA=0 to disable.
BLUR_SIGMA = 2.8                     # at 1080p; scaled automatically at other sizes
BRIGHTNESS = -0.22                   # -1 to 1; lower makes the scenery darker
SATURATION = 0.75                    # 0 = grayscale, 1 = original color
CONTRAST = 1.03

# Sizes are in pixels on a 1080p canvas, scaled automatically with TARGET_H.
SUBTITLE_FONT = "Georgia"
SUBTITLE_FONT_SIZE = 74
SUBTITLE_SIDE_MARGIN = 0.1          # fraction of width on each side; automatic line wrapping
# Use one box around the whole caption to keep translucent backgrounds uniform.
SUBTITLE_BACKGROUND_ENABLED = True
SUBTITLE_BACKGROUND_MODE = "block"   # "block" = single box (no overlap); "line" = one box per line
SUBTITLE_BACKGROUND_COLOR = "#000000" # standard #RRGGBB color, e.g. "#203040"
SUBTITLE_BACKGROUND_OPACITY = 0.2     # 0.0 = transparent, 1.0 = fully opaque
SUBTITLE_BACKGROUND_PADDING = 10.0     # pixels at 1080p around the caption (or each line in "line" mode)
WATERMARK_TEXT = ""  # set your own channel name; empty disables the watermark
WATERMARK_FONT = "Aref Ruqaa"
WATERMARK_FONT_SIZE = 45
WATERMARK_Y = 0.8                   # center of text at 87% of height (13% above bottom)
WATERMARK_ALPHA = "33"               # ASS hex transparency: 00 opaque, FF invisible

PREFER_GPU = True
PREFER_HEVC = False                  # H.264 plays in more browsers; True prefers smaller HEVC
VIDEO_CRF = 23                       # CRF on CPU, CQ on NVENC; lower = higher quality
VIDEO_PRESET = "veryfast"            # CPU preset
NVENC_PRESET = "p4"
AUDIO_BITRATE = "192k"
FFMPEG_PATH = None                   # e.g. r"C:\ffmpeg\bin\ffmpeg.exe"

# Optional silent design preview; does not need a model, translations, or media audio.
PREVIEW_SECONDS = 35.0               # nature section only; intro added only if enabled
PREVIEW_SEED = 2026

print("Nature rendering settings ready. Run Cell 8B next.")


In [ ]:
# Cell 8B: Rendering helpers. All implementation lives inside this notebook.
import gc
import hashlib
import html
import json
import math
import os
import random
import re
import secrets
import shutil
import subprocess
import tempfile
import uuid
from collections import deque
from pathlib import Path

_MEDIA_EXTS = {".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg", ".opus", ".wma",
               ".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v"}
_VIDEO_EXTS = {".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v", ".mts"}
_ROOT = Path.cwd().resolve()


def _path(value):
    p = Path(str(value).strip().strip('"').strip("'")).expanduser()
    return (p if p.is_absolute() else _ROOT / p).resolve()


def _run_ffmpeg(args, cwd=None):
    """Bounded log buffer; interrupting Jupyter also stops the child encoder."""
    tail = deque(maxlen=100)
    proc = subprocess.Popen(
        [str(_ffmpeg), "-y", "-nostdin", "-hide_banner", "-loglevel", "error", *map(str, args)],
        cwd=cwd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE,
        text=True, encoding="utf-8", errors="replace",
    )
    try:
        for line in proc.stderr:
            tail.append(line)
        if proc.wait() != 0:
            raise RuntimeError("FFmpeg failed:\n" + "".join(tail)[-6000:])
    finally:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
        proc.stderr.close()


def _find_ffmpeg():
    candidates = []
    if FFMPEG_PATH:
        candidates.append(_path(FFMPEG_PATH))
    else:
        candidates += [Path(r"C:\ffmpeg\bin\ffmpeg.exe"),
                       Path(r"C:\Program Files\ffmpeg\bin\ffmpeg.exe")]
        winget = Path.home() / "AppData/Local/Microsoft/WinGet/Packages"
        if winget.is_dir():
            candidates += sorted(winget.glob("Gyan.FFmpeg*/ffmpeg-*/bin/ffmpeg.exe"), reverse=True)
        if shutil.which("ffmpeg"):
            candidates.append(Path(shutil.which("ffmpeg")))
    for exe in candidates:
        if not exe.is_file():
            continue
        result = subprocess.run([str(exe), "-hide_banner", "-filters"], capture_output=True, text=True)
        filters = {row.split()[1] for row in result.stdout.splitlines() if len(row.split()) >= 2}
        if {"ass", "gblur", "eq", "scale", "crop", "fps"} <= filters:
            probe = exe.with_name("ffprobe.exe" if exe.suffix == ".exe" else "ffprobe")
            if not probe.is_file() and shutil.which("ffprobe"):
                probe = Path(shutil.which("ffprobe"))
            if probe.is_file():
                return exe.resolve(), probe.resolve()
    raise RuntimeError("A full FFmpeg build with libass and ffprobe is required. Set FFMPEG_PATH in Cell 8A.")


def _probe(path):
    result = subprocess.run(
        [str(_ffprobe), "-v", "error", "-show_entries",
         "format=duration:stream=codec_type,duration,width,height", "-of", "json", str(path)],
        capture_output=True, text=True, encoding="utf-8", errors="replace",
    )
    if result.returncode:
        raise RuntimeError(f"Cannot read {path.name}: {result.stderr[-1000:]}")
    return json.loads(result.stdout)


def _stream_duration(info, kind):
    streams = [s for s in info.get("streams", []) if s.get("codec_type") == kind]
    if not streams:
        raise ValueError(f"No {kind} stream found.")
    value = streams[0].get("duration", info.get("format", {}).get("duration"))
    if value in (None, "N/A"):
        value = info.get("format", {}).get("duration")
    duration = float(value)
    if not math.isfinite(duration) or duration <= 0:
        raise ValueError(f"Invalid {kind} duration: {value}")
    return duration


def _choose_encoder():
    result = subprocess.run([str(_ffmpeg), "-hide_banner", "-encoders"], capture_output=True, text=True)
    available = {p[1] for row in result.stdout.splitlines()
                 if len(p := row.split()) >= 2 and len(p[0]) == 6}
    gpu_order = ["hevc_nvenc", "h264_nvenc"] if PREFER_HEVC else ["h264_nvenc", "hevc_nvenc"]
    for name in (gpu_order if PREFER_GPU else []) + ["libx264"]:
        if name not in available:
            continue
        if name == "libx264":
            args = ["-c:v", name, "-crf", str(VIDEO_CRF), "-preset", VIDEO_PRESET]
        else:
            args = ["-c:v", name, "-preset", NVENC_PRESET, "-rc", "vbr",
                    "-cq", str(VIDEO_CRF), "-b:v", "0"]
            if name == "hevc_nvenc":
                args += ["-tag:v", "hvc1"]
        # Listed encoders can be unusable on this GPU/driver. Test before a long batch.
        try:
            _run_ffmpeg(["-f", "lavfi", "-i", f"color=s={TARGET_W}x{TARGET_H}:r={TARGET_FPS}",
                         "-frames:v", "1", *args, "-pix_fmt", "yuv420p", "-f", "null", "-"])
            print(f"Encoder: {name}")
            return name, args
        except RuntimeError as error:
            print(f"  {name} unavailable; trying the next encoder. {str(error).splitlines()[-1][:180]}")
    raise RuntimeError("No working NVENC or libx264 encoder found.")


def _nature_inventory():
    folder = _path(NATURE_FOLDER)
    if not folder.is_dir():
        raise FileNotFoundError(f"Nature folder not found: {folder}")
    clips = []
    minimum = math.ceil(MIN_CLIP_SECONDS * TARGET_FPS)
    for p in sorted(folder.rglob("*")):
        if not p.is_file() or p.suffix.lower() not in _VIDEO_EXTS:
            continue
        try:
            duration = _stream_duration(_probe(p), "video")
            # Leave one output frame of headroom to avoid seeking beyond the source.
            available = math.floor(duration * TARGET_FPS + 1e-7) - 1
            if available < minimum:
                print(f"  Skipping {p.name}: {duration:.2f}s, too short for a safe {MIN_CLIP_SECONDS:g}s excerpt.")
                continue
            clips.append({"path": str(p), "duration": duration, "available_frames": available})
        except (ValueError, RuntimeError, TypeError) as error:
            print(f"  Skipping unreadable clip {p.name}: {error}")
    if not clips:
        raise ValueError(f"No usable videos at least {MIN_CLIP_SECONDS:g}s long in {folder}")
    print(f"Usable nature clips: {len(clips)}")
    return clips


def _plan_nature(duration, clips, seed):
    """Frame-based plan: no short last shot, no audio truncation, bounded source seeks."""
    if not math.isfinite(duration) or duration <= 0:
        raise ValueError("Audio duration must be positive and finite.")
    fps = TARGET_FPS
    minimum = math.ceil(MIN_CLIP_SECONDS * fps)
    maximum = min(math.floor(MAX_CLIP_SECONDS * fps), max(c["available_frames"] for c in clips))
    if minimum > maximum:
        raise ValueError("No source is long enough for the configured minimum clip length.")
    rng = random.Random(seed)
    requested = math.ceil(duration * fps - 1e-7)
    total = max(minimum, requested)
    low_count = math.ceil(total / maximum)
    high_count = total // minimum
    if low_count > high_count:
        # E.g. a very narrow 10-11s range cannot cover 12s; pad rather than cut audio.
        total = low_count * minimum
        high_count = total // minimum
    count = rng.randint(low_count, high_count)
    remaining = total
    lengths = []
    for i in range(count):
        left = count - i - 1
        lower = max(minimum, remaining - left * maximum)
        upper = min(maximum, remaining - left * minimum)
        length = rng.randint(lower, upper)
        lengths.append(length)
        remaining -= length
    rng.shuffle(lengths)
    plan, uses, previous, elapsed = [], {}, None, 0
    for frames in lengths:
        eligible = [c for c in clips if c["available_frames"] >= frames]
        alternatives = [c for c in eligible if c["path"] != previous]
        if alternatives:
            eligible = alternatives
        least_used = min(uses.get(c["path"], 0) for c in eligible)
        clip = rng.choice([c for c in eligible if uses.get(c["path"], 0) == least_used])
        start_frame = rng.randint(0, clip["available_frames"] - frames)
        plan.append({"source": clip["path"], "source_duration": clip["duration"],
                     "source_start": start_frame / fps, "source_end": (start_frame + frames) / fps,
                     "frames": frames, "duration": frames / fps, "timeline_start": elapsed / fps})
        uses[clip["path"]] = uses.get(clip["path"], 0) + 1
        previous = clip["path"]
        elapsed += frames
    return plan


_SRT_TIME = r"(\d+):(\d{2}):(\d{2})[,.](\d{3})"


def _read_srt(path):
    cues = []
    for block in re.split(r"\n\s*\n", path.read_text(encoding="utf-8-sig").strip()):
        lines = block.splitlines()
        at = 1 if lines and lines[0].strip().isdigit() else 0
        if len(lines) <= at:
            continue
        match = re.fullmatch(_SRT_TIME + r"\s*-->\s*" + _SRT_TIME, lines[at].strip())
        if not match:
            raise ValueError(f"Invalid SRT timing in {path.name}: {lines[at]!r}")
        values = list(map(int, match.groups()))
        def seconds(v):
            h, m, s, ms = v
            return h * 3600 + m * 60 + s + ms / 1000
        start, end = seconds(values[:4]), seconds(values[4:])
        text = "\n".join(lines[at + 1:]).strip()
        if end <= start:
            raise ValueError(f"Subtitle ends before it starts in {path.name}: {lines[at]}")
        if text:
            cues.append((start, end, text))
    if not cues:
        raise ValueError(f"No captions in {path}")
    return cues


def _ass_time(seconds):
    centis = max(0, round(seconds * 100))
    hours, rem = divmod(centis, 360000)
    minutes, rem = divmod(rem, 6000)
    secs, centis = divmod(rem, 100)
    return f"{hours}:{minutes:02d}:{secs:02d}.{centis:02d}"


def _ass_text(text):
    # Escape literal ASS control characters; retain common SRT italic/bold markup.
    text = html.unescape(text).replace("\\", "\\\\").replace("{", r"\{").replace("}", r"\}")
    for tag in ("i", "b", "u"):
        text = re.sub(fr"<{tag}>", lambda _: "{\\" + tag + "1}", text, flags=re.I)
        text = re.sub(fr"</{tag}>", lambda _: "{\\" + tag + "0}", text, flags=re.I)
    return text.replace("\r", "").replace("\n", r"\N")


def _write_ass(srt, dest, intro_seconds, total_seconds):
    scale = TARGET_H / 1080
    margin = round(TARGET_W * SUBTITLE_SIDE_MARGIN)
    if SUBTITLE_BACKGROUND_ENABLED:
        rgb = SUBTITLE_BACKGROUND_COLOR.lstrip("#")
        red, green, blue = (int(rgb[i:i + 2], 16) for i in (0, 2, 4))
        alpha = round((1 - SUBTITLE_BACKGROUND_OPACITY) * 255)
        outline_color = back_color = f"&H{alpha:02X}{blue:02X}{green:02X}{red:02X}"
        if SUBTITLE_BACKGROUND_MODE == "block":
            # libass BorderStyle=4 draws one background for the entire caption,
            # including automatically wrapped lines. BackColour sets its alpha
            # and color; Shadow sets padding, without an offset or text shadow.
            # No glyph outline: each background pixel receives alpha only once.
            border_style, outline, shadow = 4, 0.0, SUBTITLE_BACKGROUND_PADDING * scale
        else:
            # Original BorderStyle=3: one box per line, which may overlap.
            border_style, outline, shadow = 3, SUBTITLE_BACKGROUND_PADDING * scale, 0.0
    else:
        outline_color, back_color = "&HA0000000", "&H80000000"
        border_style, outline, shadow = 1, 0.5 * scale, 0.8 * scale
    text = f"""[Script Info]
ScriptType: v4.00+
PlayResX: {TARGET_W}
PlayResY: {TARGET_H}
WrapStyle: 0
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Caption,{SUBTITLE_FONT},{SUBTITLE_FONT_SIZE * scale:.2f},&H00FFFFFF,&H00FFFFFF,{outline_color},{back_color},0,0,0,0,100,100,0,0,{border_style},{outline:.2f},{shadow:.2f},5,{margin},{margin},0,1
Style: Watermark,{WATERMARK_FONT},{WATERMARK_FONT_SIZE * scale:.2f},&H{WATERMARK_ALPHA}FFFFFF,&H00FFFFFF,&HA0000000,&H80000000,0,0,0,0,100,100,1,0,1,0.4,0.5,5,0,0,0,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
    for start, end, caption in _read_srt(srt):
        text += (f"Dialogue: 1,{_ass_time(start + intro_seconds)},{_ass_time(end + intro_seconds)},"
                 f"Caption,,0,0,0,,{_ass_text(caption)}\n")
    if WATERMARK_TEXT:
        pos = "{\\pos(" + f"{TARGET_W / 2:g},{TARGET_H * WATERMARK_Y:g}" + ")}"
        text += (f"Dialogue: 2,{_ass_time(intro_seconds)},{_ass_time(total_seconds)},"
                 f"Watermark,,0,0,0,,{pos}{_ass_text(WATERMARK_TEXT)}\n")
    dest.write_text(text, encoding="utf-8")


def _normalized_video():
    return (f"scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=increase,"
            f"crop={TARGET_W}:{TARGET_H},setsar=1,fps={TARGET_FPS},format=yuv420p")


def _encode_part(source, dest, frames, work_dir, *, start=0.0, timeline=None, fade=0.0):
    vf = (_normalized_video() + ",tpad=stop_mode=clone:stop_duration=1,"
          f"trim=end_frame={frames},setpts=N/({TARGET_FPS}*TB)")
    if timeline is not None:
        if BLUR_SIGMA:
            vf += f",gblur=sigma={BLUR_SIGMA * TARGET_H / 1080:.4f}:steps=2"
        vf += f",eq=brightness={BRIGHTNESS}:saturation={SATURATION}:contrast={CONTRAST}"
        # Captions use global time, then each encoded part restarts at zero for concat.
        vf += f",setpts=PTS+{timeline:.9f}/TB,ass=filename=captions.ass,setpts=PTS-STARTPTS"
        if fade > 0:
            vf += f",fade=t=out:st={frames / TARGET_FPS - fade:.9f}:d={fade:.9f}"
    _run_ffmpeg(["-ss", f"{start:.9f}", "-i", str(source), "-map", "0:v:0", "-an",
                 "-vf", vf, "-frames:v", str(frames), *_encode_args,
                 "-pix_fmt", "yuv420p", "-r", str(TARGET_FPS), "-video_track_timescale", "24000",
                 "-map_metadata", "-1", str(dest)], cwd=work_dir)
    # A truncated/changed source must fail, rather than silently shorten the timeline.
    actual = _stream_duration(_probe(dest), "video")
    if abs(actual - frames / TARGET_FPS) > 0.5 / TARGET_FPS:
        raise RuntimeError(f"Unexpected encoded duration for {dest.name}: {actual:.6f}s")


def _render_nature(audio, srt, output, *, clips, seed, limit_seconds=None):
    audio, srt, output = Path(audio).resolve(), Path(srt).resolve(), Path(output).resolve()
    audio_duration = _stream_duration(_probe(audio), "audio")
    if limit_seconds is not None:
        audio_duration = min(audio_duration, limit_seconds)
    plan = _plan_nature(audio_duration, clips, seed)
    body_seconds = sum(p["frames"] for p in plan) / TARGET_FPS
    intro = _path(INTRO_PATH) if INCLUDE_INTRO and INTRO_PATH else None
    intro_info = _probe(intro) if intro else None
    intro_frames = math.ceil(_stream_duration(intro_info, "video") * TARGET_FPS - 1e-7) if intro else 0
    intro_seconds = intro_frames / TARGET_FPS
    total_seconds = intro_seconds + body_seconds
    output.parent.mkdir(parents=True, exist_ok=True)
    token = uuid.uuid4().hex[:10]
    partial = output.with_name(f"{output.stem}.{token}.part.mp4")
    manifest = output.with_suffix(".clips.json")
    manifest_part = manifest.with_name(f"{manifest.name}.{token}.part")
    print(f"  Audio: {audio_duration:.2f}s | nature: {body_seconds:.2f}s | intro: {intro_seconds:.2f}s")
    print(f"  {len(plan)} randomized excerpts | seed: {seed} | output: {output.name}")
    try:
        # One source decoded at a time, so long videos do not open hundreds of 4K decoders.
        with tempfile.TemporaryDirectory(prefix="_nature_", dir=output.parent) as temp:
            work = Path(temp)
            _write_ass(srt, work / "captions.ass", intro_seconds, total_seconds)
            parts = []
            if intro:
                dest = work / "part_00000.mp4"
                print("  Encoding intro...", flush=True)
                _encode_part(intro, dest, intro_frames, work)
                parts.append((dest, intro_seconds))
            for i, shot in enumerate(plan, 1):
                dest = work / f"part_{i:05d}.mp4"
                print(f"  [{i}/{len(plan)}] {Path(shot['source']).name}: "
                      f"{shot['source_start']:.2f}-{shot['source_end']:.2f}s "
                      f"({shot['duration']:.2f}s)", flush=True)
                fade = min(FADE_SECONDS, shot["duration"]) if i == len(plan) else 0.0
                _encode_part(shot["source"], dest, shot["frames"], work,
                             start=shot["source_start"], timeline=intro_seconds + shot["timeline_start"], fade=fade)
                parts.append((dest, shot["duration"]))
            # All parts share codec, canvas, frame rate, pixel format and time base.
            # Generated ASCII basenames avoid Windows filter/concat escaping issues.
            (work / "parts.ffconcat").write_text(
                "ffconcat version 1.0\n" + "".join(
                    f"file {p.name}\nduration {seconds:.9f}\n" for p, seconds in parts), encoding="utf-8")
            args = ["-f", "concat", "-safe", "1", "-i", "parts.ffconcat", "-i", str(audio)]
            main = (f"[1:a:0]atrim=duration={audio_duration:.9f},asetpts=PTS-STARTPTS,"
                    "aresample=48000,aformat=sample_fmts=fltp:channel_layouts=stereo[main];")
            if intro:
                has_audio = any(s.get("codec_type") == "audio" for s in intro_info["streams"])
                if has_audio:
                    args += ["-i", str(intro)]
                    head = (f"[2:a:0]atrim=duration={intro_seconds:.9f},asetpts=PTS-STARTPTS,"
                            "aresample=48000,aformat=sample_fmts=fltp:channel_layouts=stereo,"
                            f"apad,atrim=duration={intro_seconds:.9f}[head];")
                else:
                    head = f"anullsrc=r=48000:cl=stereo,atrim=duration={intro_seconds:.9f}[head];"
                af = main + head + "[head][main]concat=n=2:v=0:a=1[joined];"
            else:
                af = main + "[main]anull[joined];"
            af += f"[joined]apad,atrim=duration={total_seconds:.9f}[aout]"
            _run_ffmpeg([*args, "-filter_complex", af, "-map", "0:v:0", "-map", "[aout]",
                         "-c:v", "copy", "-c:a", "aac", "-b:a", AUDIO_BITRATE,
                         "-t", f"{total_seconds:.9f}", "-movflags", "+faststart", str(partial)], cwd=work)
            result_info = _probe(partial)
            if abs(_stream_duration(result_info, "video") - total_seconds) > 1 / TARGET_FPS:
                raise RuntimeError("Final video duration does not match the planned timeline.")
            if abs(_stream_duration(result_info, "audio") - total_seconds) > 0.1:
                raise RuntimeError("Final audio duration does not match the planned timeline.")
            details = {
                "seed": seed, "audio": str(audio), "srt": str(srt),
                "audio_seconds": audio_duration, "intro": str(intro) if intro else None,
                "intro_seconds": intro_seconds, "nature_seconds": body_seconds, "total_seconds": total_seconds,
                "canvas": [TARGET_W, TARGET_H], "fps": TARGET_FPS, "encoder": _encoder,
                "settings": {name: globals()[name] for name in (
                    "MIN_CLIP_SECONDS", "MAX_CLIP_SECONDS", "BLUR_SIGMA", "BRIGHTNESS", "SATURATION",
                    "CONTRAST", "FADE_SECONDS", "INCLUDE_INTRO", "SUBTITLE_FONT", "SUBTITLE_FONT_SIZE", "SUBTITLE_SIDE_MARGIN",
                    "SUBTITLE_BACKGROUND_ENABLED", "SUBTITLE_BACKGROUND_COLOR", "SUBTITLE_BACKGROUND_OPACITY",
                    "SUBTITLE_BACKGROUND_PADDING", "SUBTITLE_BACKGROUND_MODE",
                    "WATERMARK_TEXT", "WATERMARK_FONT", "WATERMARK_FONT_SIZE", "WATERMARK_Y", "WATERMARK_ALPHA")},
                "clips": plan,
            }
            manifest_part.write_text(json.dumps(details, ensure_ascii=False, indent=2), encoding="utf-8")
            os.replace(partial, output)
            os.replace(manifest_part, manifest)
    finally:
        partial.unlink(missing_ok=True)
        manifest_part.unlink(missing_ok=True)
    print(f"  Saved {output} ({output.stat().st_size / 1e6:.1f} MB)")
    return output


def _prepare_nature():
    global _ffmpeg, _ffprobe, _encoder, _encode_args
    if not (isinstance(TARGET_FPS, int) and TARGET_FPS > 0):
        raise ValueError("TARGET_FPS must be a positive integer.")
    if TARGET_W <= 0 or TARGET_H <= 0 or TARGET_W % 2 or TARGET_H % 2:
        raise ValueError("Output width/height must be positive even numbers.")
    if not (0 < MIN_CLIP_SECONDS <= MAX_CLIP_SECONDS):
        raise ValueError("Use 0 < MIN_CLIP_SECONDS <= MAX_CLIP_SECONDS.")
    if not (0 <= WATERMARK_Y <= 0.95 and 0 <= SUBTITLE_SIDE_MARGIN < 0.5):
        raise ValueError("Keep WATERMARK_Y between 0 and 0.95, and side margin below 0.5.")
    if not (BLUR_SIGMA >= 0 and FADE_SECONDS >= 0 and -1 <= BRIGHTNESS <= 1
            and 0 <= SATURATION <= 3 and 0 <= CONTRAST <= 3):
        raise ValueError("Invalid blur, fade, brightness, saturation or contrast setting.")
    if not re.fullmatch(r"[0-9a-fA-F]{2}", WATERMARK_ALPHA):
        raise ValueError("WATERMARK_ALPHA must be two hex digits, e.g. '40'.")
    if not re.fullmatch(r"#[0-9a-fA-F]{6}", SUBTITLE_BACKGROUND_COLOR):
        raise ValueError("SUBTITLE_BACKGROUND_COLOR must be a #RRGGBB color, e.g. '#000000'.")
    if SUBTITLE_BACKGROUND_MODE not in {"block", "line"}:
        raise ValueError("SUBTITLE_BACKGROUND_MODE must be 'block' (one box) or 'line' (per-line boxes).")
    if not (0 <= SUBTITLE_BACKGROUND_OPACITY <= 1 and math.isfinite(SUBTITLE_BACKGROUND_PADDING)
            and SUBTITLE_BACKGROUND_PADDING >= 0):
        raise ValueError("Use background opacity from 0.0 to 1.0 and nonnegative padding.")
    for font in (SUBTITLE_FONT, WATERMARK_FONT):
        if any(char in font for char in ",\n\r"):
            raise ValueError("Font names cannot contain commas or line breaks.")
    if INCLUDE_INTRO and INTRO_PATH and not _path(INTRO_PATH).is_file():
        raise FileNotFoundError(f"Intro not found: {_path(INTRO_PATH)}. Set INCLUDE_INTRO=False to omit it.")
    # Release the transcription model before testing NVENC / starting a long batch.
    globals().pop("model", None)
    gc.collect()
    try:
        import sys
        torch_module = sys.modules.get("torch")
        if torch_module is not None and torch_module.cuda.is_available():
            torch_module.cuda.empty_cache()
    except Exception:
        pass
    _ffmpeg, _ffprobe = _find_ffmpeg()
    print(f"FFmpeg: {_ffmpeg}")
    _encoder, _encode_args = _choose_encoder()
    return _nature_inventory()


print("Nature helpers ready. Run Cell 8C for a silent preview, or Cell 8D for final videos.")


### Cell 8C — Optional silent preview

Run 8A and 8B first, then this cell. This generates a 35-second nature sample
(plus the configured intro only if enabled), with clearly labeled sample captions and silent audio
during the nature section. It does not read or modify your translations.
The preview is saved to `output/nature/preview/nature_style_preview.mp4`.
Adjust 8A, then rerun this preview cell to compare the appearance.
Set `PREFER_HEVC=False` if your notebook/browser cannot play HEVC.
The sample caption timing is fixed at 0–35 seconds; changing `PREVIEW_SECONDS`
changes only the preview length. Final-video captions come from your translations.


In [ ]:
# Cell 8C (optional): Silent design preview with sample captions.
# Run 8A + 8B + this cell directly; no Whisper download or translation is needed.
from IPython.display import Video, display

clips = _prepare_nature()
preview_dir = _path(OUTPUT_FOLDER) / "preview"
preview_dir.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryDirectory(prefix="_preview_", dir=preview_dir) as temp:
    temp = Path(temp)
    demo_audio = temp / "silent.wav"
    demo_srt = temp / "demo.srt"
    _run_ffmpeg(["-f", "lavfi", "-i", "anullsrc=r=48000:cl=stereo", "-t", str(PREVIEW_SECONDS),
                 "-c:a", "pcm_s16le", str(demo_audio)])
    demo_srt.write_text(
        "1\n00:00:00,000 --> 00:00:09,000\nYour translated subtitles will appear here.\n\n"
        "2\n00:00:09,000 --> 00:00:20,000\nA quiet moment to reflect,\nand a reminder for the heart.\n\n"
        "3\n00:00:20,000 --> 00:00:35,000\nSample captions — replace these with your translation.\n",
        encoding="utf-8",
    )
    PREVIEW_RESULT = _render_nature(
        demo_audio, demo_srt, preview_dir / "nature_style_preview.mp4", clips=clips, seed=PREVIEW_SEED)
print("Silent sample only; these captions are not a translation of your audio.")
display(Video(filename=str(PREVIEW_RESULT), embed=True, width=960))


### Cell 8D — Batch final videos

Run 8A and 8B first. The optional preview is not required. This cell uses the
original transcript-to-SRT conversion, then renders every translated media pair.
Your English files must contain `[MM:SS-MM:SS]` lines, as in the original notebook.
English stubs that are still blank cannot be rendered; the cell reports this and
points you to the preview. If you change the visual settings, set
`OVERWRITE_FINAL=True` to replace existing nature exports.

**Skipping completed exports:** with `OVERWRITE_FINAL=False`, the batch skips a
pair if either `<name>_final.mp4` or `<name>_final.clips.json` is present in
`OUTPUT_FOLDER`. Keep the completion JSON when you rename a finished video so
the batch remembers it. Failed exports without either file are retried.
`OVERWRITE_FINAL=True` bypasses both checks to force a fresh render.


In [ ]:
# Cell 8D: Same filename-matched batch workflow, with the new nature rendering.
# Run 8A + 8B first. Cell 8C is optional.

from transcriber_utils import convert_transcript_file as _txt_to_srt, pair_transcripts

media_dir = _path(MEDIA_FOLDER)
transcript_dir = _path(TRANSCRIPT_FOLDER)
output_dir = _path(OUTPUT_FOLDER)
for folder in (media_dir, transcript_dir):
    if not folder.is_dir():
        raise FileNotFoundError(f"Folder not found: {folder}")
output_dir.mkdir(parents=True, exist_ok=True)

pairs, blank, unmatched = pair_transcripts(media_dir, transcript_dir, only=ONLY)

print(f"Pairs: {len(pairs)} | blank translations: {len(blank)} | unmatched: {len(unmatched)}")
for name in unmatched:
    print(f"  No matching media: {name}")
if not pairs:
    raise FileNotFoundError(
        "No translated transcript/media pairs found. Fill transcript/english/<name>.txt with "
        "[MM:SS-MM:SS] English lines, matching input/<name>.mp3 (or another media extension). "
        "For a design preview while translations are blank, run Cell 8C."
    )

# A completed .clips.json record remains useful after the exported video is renamed.
# Records are written only after a successful render. Temporary .part files do not count.
pending, skipped = [], []
for txt, media in pairs:
    final_video = output_dir / f"{media.stem}_final.mp4"
    completed_record = final_video.with_suffix(".clips.json")
    if not OVERWRITE_FINAL and (final_video.is_file() or completed_record.is_file()):
        reason = "video exists" if final_video.is_file() else "completed .clips.json record exists"
        print(f"  Skipped: {media.name} — {reason}")
        skipped.append(media.name)
    else:
        pending.append((txt, media))
done, failed = [], []
if pending:
    clips = _prepare_nature()
    batch_seed = RANDOM_SEED if RANDOM_SEED is not None else secrets.randbits(64)
    print(f"Batch random seed: {batch_seed}")
    for idx, (txt, media) in enumerate(pending, 1):
        print(f"\n[{idx}/{len(pending)}] {media.name}", flush=True)
        try:
            srt = _txt_to_srt(txt)
            # A stable per-file seed means processing order and skipped files do not change its mix.
            seed = int.from_bytes(hashlib.sha256(f"{batch_seed}:{media.stem}".encode("utf-8")).digest()[:8], "big")
            _render_nature(media, srt, output_dir / f"{media.stem}_final.mp4", clips=clips, seed=seed)
            done.append(media.name)
        except Exception as error:
            failed.append(media.name)
            print(f"  FAILED: {error}")

print(f"\nRendered: {len(done)} | skipped existing: {len(skipped)} | failed: {len(failed)}")
for name in failed:
    print(f"  Failed: {name}")
print(f"Final videos: {output_dir}")

if failed:
    raise RuntimeError(f"Rendering failed for: {', '.join(failed)}")


## 📝 Convert transcripts to SRT

**Cell 9** converts timestamped transcripts (`[MM:SS-MM:SS]` lines) into `.srt` subtitle files, using the same conversion as Cell 8 — without rendering video.

Point `TRANSCRIPT_FOLDER` at a folder (default `transcript/english/`). Every non-blank `<name>.txt` becomes `<name>.srt` next to it (`1.txt` → `1.srt`). Blank English stubs are skipped.

Set `ONLY` to a stem (e.g. `"1"`) to convert a single file.

In [ ]:
# Cell 9: Convert translated timestamped files into SRT (no rendering).
from pathlib import Path
from transcriber_utils import convert_transcript_file, list_transcripts
TRANSCRIPT_FOLDER = "transcript/english"
ONLY = None  # one filename stem, or None for all
files = list_transcripts(TRANSCRIPT_FOLDER)
if ONLY is not None:
    files = [p for p in files if p.stem == str(ONLY)]
if not files:
    raise FileNotFoundError("No matching transcripts. Complete the transcription/translation steps first.")
for txt in files:
    if not txt.read_text(encoding="utf-8-sig").strip():
        print(f"Skipped blank translation: {txt.name}")
        continue
    print(f"Wrote {convert_transcript_file(txt)}")
